# Task 2 — 耳朵和头的形状变了，HRTF 怎么变？（第一版：有名字的形状量）

**2026-10-11 建。探索性。** Task 1 问"能不能从几何预测一个人的 HRTF"；Task 2 问"形状和 HRTF 之间是什么样的关系"：
某个形状方向上人与人不同，HRTF 在哪些方向、哪些频率、差多少 dB；这个关系在仿真和实测里是不是同一个。

**先浅后深。** 这一版只用已经对应好、能直接解释的形状量；SONICOM 的稠密耳廓对应（非刚性配准）留到后面，看时间。

| | SONICOM（164 人） | SYMARE（46 人） |
|---|---|---|
| 形状输入 | FFM_3_2_1 的 9 个简单量：**头**（两耳道间距的一半、头长、头顶高）+ **耳**（耳道前后 / 上下位置、耳廓外突、耳廓高、耳廓前后长、耳部面积）；按列标准化 | 共同头坐标系里的 landmark（mm，不标准化）：**头**（中矢面 5 点 + 两眼）+ **耳**（同侧耳廓 13 点） |
| HRTF | 仿真和实测的 DTF，与 Task 1 相同（dB，bin 1–85，仰角 ≥ −45° 的 2652 个方向，每只耳每个 bin 减去方向平均），**不经过 AE** | 同左 |

每只耳一行，右耳的形状和 HRTF 都镜像到左耳；交叉验证、bootstrap、置换都按人。折与 Task 1 相同。锁定的测试受试者不读。

## 设计（跑之前写死）

1. **复现检查**：Task 1 的几何特征 → 实测 DTF 必须回到 Task 1 的数（SONICOM 0.0693，SYMARE 0.0509，±0.002）。
2. **预测**：线性 ridge（Task 1 的引擎），头 / 耳 / 全部 → 仿真、实测，CV-R²（总体和三个频段）；全部输入的降秩版本（秩 1–3）；
   全部输入做 100 次按人置换；bootstrap：耳在头之上多出来的 R²，仿真 − 实测。
3. **模式（map）**：在全部开发集上拟合 形状 → DTF 的 ridge 映射（α = "全部 → 仿真" 10 个外折所选 α 的中位数，**两个域用同一个 α**，
   这样两边的收缩相同）。对**仿真**映射的拟合值做 SVD（降秩回归的模式）：模式 k = 一个形状得分；
   **形状变化** $p_k$ = 得分 +1 SD 时人群里平均的形状变化（mm）；**HRTF 变化** = 两个映射对同一个形状变化 $p_k$ 的响应
   $r_{\text{sim}}=B_{\text{sim}}^\top p_k$、$r_{\text{meas}}=B_{\text{meas}}^\top p_k$（dB，每个方向 × 频率）。
   - 比较：两个响应的余弦（模式是否相同）、rms 之比 meas / sim（保留了多少），总体和分频段。
   - 统计：余弦 1000 次置换（把实测按人打乱）和 1000 次按人 bootstrap；仿真第 1 个模式的强度做 1000 次置换；
     模式稳定性 = bootstrap 重新求出的形状变化与原来的 |余弦|。
4. **读法**（写死）：仿真模式 1 不比置换强 → 这些输入和 N 下没有可靠的仿真模式（原因 A）。
   某个模式余弦的置换 p < 0.05 且 CI 在 0 以上 → 这个形状–HRTF 关系在实测里也存在，比值说明保留多少；否则这个关系在实测里看不到。
   只看前 3 个模式；两个库各自报告，不做多重比较校正，结论按探索性写。
5. **范围**：这是人与人之间的关联，不是同一个人改耳朵时 HRTF 的导数（那需要改网格重算 BEM，未来工作）。

## Drive 上要有的文件
与 Task 1 相同：`ELEC5021/data_48k_2026-10-10.zip`、`ELEC5021/split_48k.json`、`ELEC5020/SONICOM/mesh2z_samples.zip`、
`HRTF_Checkpoints/mesh2z_sonicom_latents_v1.npz`、`HRTF_Checkpoints/mesh2z_rff_v1/oof_ridge.npz`、`ELEC5021/SYMARE/3DShapeData.mat`。
结果写到 `HRTF_Checkpoints/task2_v1/`。CPU 运行时即可（有 GPU 也行），从上到下跑。

In [ ]:
!pip install healpy -q

In [ ]:
# ==========================================================================
# Cell 0 — 路径 · 数据（zip 按 SHA-256 核对后只解压要用的文件夹；读到的每个数组再按 manifest 核对）
# ==========================================================================
import torch
import os, json, csv, time, hashlib, zipfile, shutil
import numpy as np
import healpy as hp
import scipy.io as sio
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'
SAVE_DIR = f'{DRIVE}/HRTF_Checkpoints'
PATHS = dict(
    data_zip    = f'{DRIVE}/ELEC5021/data_48k_2026-10-10.zip',
    split       = f'{DRIVE}/ELEC5021/split_48k.json',
    samples_zip = f'{DRIVE}/ELEC5020/SONICOM/mesh2z_samples.zip',
    old_latents = f'{SAVE_DIR}/mesh2z_sonicom_latents_v1.npz',
    old_oof     = f'{SAVE_DIR}/mesh2z_rff_v1/oof_ridge.npz',
    landmarks   = f'{DRIVE}/ELEC5021/SYMARE/3DShapeData.mat',
)
for k, p in PATHS.items():
    print(f'{k:12s} {"OK " if os.path.exists(p) else "★缺失"} {p}')
assert all(os.path.exists(p) for p in PATHS.values()), '先把缺失的文件放好（与 Task 1 相同）'
ZIP_SHA256, SPLIT_SHA256, SAMPLES_SHA256 = 'f51c6b072123e70d6d8a0691373ab90e3f6db9856ce6306f65d949c87fbf97c7', 'dc89bc454fef351951283cccb3772fdd3e552bbdde5d13668f8f576f4cd83561', '8340bdf73be1a8722015e877bcca2957570293484277989f1245eb8d65aa085a'
OUT = f'{SAVE_DIR}/task2_v1'
os.makedirs(OUT, exist_ok=True)


def sha256(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def stage(zpath, digest, dest, keep=lambda n: True):
    stamp = f'{dest}/.zip_ok_{digest[:16]}'
    if not os.path.exists(stamp):
        t0 = time.time()
        assert sha256(zpath) == digest, f'{os.path.basename(zpath)} 的 SHA-256 不符'
        shutil.rmtree(dest, ignore_errors=True); os.makedirs(dest)
        with zipfile.ZipFile(zpath) as z:
            z.extractall(dest, [n for n in z.namelist() if keep(n)])
        open(stamp, 'w').write(digest)
        print(f'{os.path.basename(zpath)} 核对并解压 ({time.time() - t0:.0f}s)')


KEEP = tuple(f'data_48k/{t}_48k/' for t in ('SONICOM', 'SONICOM_SIM', 'SYMARE', 'SYMARE_BEM'))
stage(PATHS['data_zip'], ZIP_SHA256, '/content/task2_data', lambda n: n.startswith(KEEP) or n == 'data_48k/healpix_mirror_idx.npy')
stage(PATHS['samples_zip'], SAMPLES_SHA256, '/content/mesh2z_samples_local')
DATA = '/content/task2_data/data_48k'
SAMPLES = '/content/mesh2z_samples_local/mesh2z_samples'
assert os.path.exists(f'{SAMPLES}/manifest.csv')
assert sha256(PATHS['split']) == SPLIT_SHA256, 'split_48k.json 的 SHA-256 不符'
SPLIT = json.load(open(PATHS['split']))
MIRROR = np.load(f'{DATA}/healpix_mirror_idx.npy')
assert np.array_equal(MIRROR[MIRROR], np.arange(3072))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_grad_enabled(False)
print('device:', device)

In [ ]:
# ==========================================================================
# Cell 1 — 设定（跑之前写死，不要改）
# ==========================================================================
N_BINS, FS = 86, 48000
FREQS = np.arange(N_BINS) * FS / 256
DTF_F = FREQS[1:]                                          # DTF 用 bin 1–85
BANDS = {'0.2-5 kHz': (200, 5000), '6-12 kHz': (6000, 12000), '12-16 kHz': (12000, 16000)}
BAND_DTF = {b: (DTF_F >= lo) & (DTF_F <= hi) for b, (lo, hi) in BANDS.items()}
ALPHA_REL = list(np.logspace(-4, 5, 37))
N_FOLDS, N_INNER, SEED = 10, 5, 0
N_PERM_CV, N_PERM, N_BOOT, N_MODES = 100, 1000, 1000, 3
SIGMAS, M_RFF = [0.005, 0.01, 0.02, 0.05, 0.1], 256        # 只用于复现检查（Task 1 的 SONICOM 几何特征）
GATE = {"SONICOM": 0.06929722198908661, "SYMARE": 0.05087221218036431}                                   # Task 1：geometry -> DTF_meas（results/task1_results.json）

_th, _ph = hp.pix2ang(16, np.arange(3072))
EL = 90 - np.degrees(_th)
AZ = (np.degrees(_ph) + 180) % 360 - 180                    # (-180, 180]，+ = 左（同侧，右耳已镜像）
MEAS_DIRS = EL >= -45.0
DIRS = np.flatnonzero(MEAS_DIRS)
POS = {p: j for j, p in enumerate(DIRS)}
# 水平面：赤道环（仰角正好 0°，64 个像素）；正中面：每个环取最靠近正前和正后的像素
_h = DIRS[np.abs(EL[DIRS]) < 1e-9]
H_PLANE = [(float(AZ[p]), POS[p]) for p in _h[np.argsort(AZ[_h])]]
_med = []
for e in np.unique(EL[DIRS]):
    ring = DIRS[EL[DIRS] == e]
    _med.append((float(e), POS[ring[np.argmin(np.abs(AZ[ring]))]]))
    _med.append((float(180 - e), POS[ring[np.argmin(180 - np.abs(AZ[ring]))]]))
M_PLANE = sorted(_med)
assert len(H_PLANE) == 64
print(f'DTF：{len(DTF_F)} 个 bin × {len(DIRS)} 个方向 | 水平面 {len(H_PLANE)} 个方向，正中面 {len(M_PLANE)} 个（−45° 到 225°）')

In [ ]:
# ==========================================================================
# Cell 2 — 回归引擎：核 ridge 的对偶形式（与 FFM_3_2_1 Cell 6 的预测完全相同）
#
# FFM_3_2_1 在每折内对目标做 PCA 只是为了快；这里直接用目标的 Gram 矩阵 Y Yᵀ 算误差，
# 对任意维度的目标（latent 5504 维、DTF 约 22 万维）都一样快。
# ==========================================================================
def subject_folds(subjects, k, rng):
    p = rng.permutation(subjects)
    return [np.sort(a) for a in np.array_split(p, min(k, len(p)))]


def make_plan(ns, seed=SEED):
    # FFM_3_2_1 的 build_plan：随机数的调用顺序相同，所以折相同
    rng = np.random.default_rng(seed)
    subj = np.repeat(np.arange(ns), 2)
    rows = lambda s: np.flatnonzero(np.isin(subj, s))
    plan = []
    for te in subject_folds(np.arange(ns), N_FOLDS, rng):
        tr = np.setdiff1d(np.arange(ns), te)
        inner = [(rows(np.setdiff1d(tr, v)), rows(v)) for v in subject_folds(tr, N_INNER, rng)]
        plan.append(((rows(tr), rows(te)), inner))
    return plan


def centred_pair(G, tr, va):
    Gtt, Gvt = G[np.ix_(tr, tr)], G[np.ix_(va, tr)]
    cm, tot = Gtt.mean(0), Gtt.mean()
    return Gtt - cm[None] - cm[:, None] + tot, Gvt - cm[None] - Gvt.mean(1)[:, None] + tot


def centred_vv_diag(G, tr, va):
    r = G[np.ix_(va, tr)].mean(1)
    return np.diag(G[np.ix_(va, va)]) - 2 * r + G[np.ix_(tr, tr)].mean()


def weights(K, tr, va, alphas):
    Ktt, Kvt = centred_pair(K, tr, va)
    lam, Q = np.linalg.eigh(Ktt)
    lam = np.clip(lam, 0, None)
    KvQ, scale = Kvt @ Q, np.trace(Ktt) / len(tr)
    return [(KvQ / (lam + a * scale)) @ Q.T for a in alphas]


def row_sse(W, G, tr, va):
    CYtt, CYvt = centred_pair(G, tr, va)
    return centred_vv_diag(G, tr, va) - 2 * np.einsum('ij,ij->i', W, CYvt) + np.einsum('ij,ij->i', W @ CYtt, W)


def run_cv(K, T, plan, keep_pred=False, parts=True):
    # T: {'G': Y Yᵀ, 'parts': {名字: 部分目标的 Gram}, 'Y': 显式目标（要预测值时）}
    n = K.shape[0]
    sse, sst = np.zeros(n), np.zeros(n)
    psse = {k: np.zeros(n) for k in T['parts']} if parts else {}
    psst = {k: np.zeros(n) for k in T['parts']} if parts else {}
    pred = np.zeros_like(T['Y']) if keep_pred else None
    chosen = []
    for (tr, va), inner in plan:
        tot = np.zeros(len(ALPHA_REL))
        for itr, iva in inner:
            for j, W in enumerate(weights(K, itr, iva, ALPHA_REL)):
                tot[j] += row_sse(W, T['G'], itr, iva).sum()
        j = int(np.argmin(tot)); chosen.append(j)
        W = weights(K, tr, va, [ALPHA_REL[j]])[0]
        sse[va] = row_sse(W, T['G'], tr, va); sst[va] = centred_vv_diag(T['G'], tr, va)
        for k, G in (T['parts'].items() if parts else ()):
            psse[k][va] = row_sse(W, G, tr, va); psst[k][va] = centred_vv_diag(G, tr, va)
        if keep_pred:
            Ytr = T['Y'][tr]; my = Ytr.mean(0)
            pred[va] = my + W @ (Ytr - my)
    return dict(r2=float(1 - sse.sum() / sst.sum()), sse=sse, sst=sst, alpha_idx=chosen, pred=pred,
                parts={k: float(1 - psse[k].sum() / psst[k].sum()) for k in psse})


gram = lambda Y: Y @ Y.T


def zscore(X):
    sd = X.std(0)
    keep = sd > 1e-12 * max(sd.max(), 1e-300)
    return (X[:, keep] - X[:, keep].mean(0)) / sd[keep]


def unit_trace(G):
    n = len(G); C = np.eye(n) - 1.0 / n
    return G / (np.trace(C @ G @ C) / n)


def permuted(G, perm):
    r = np.array([2 * perm[k] + s for k in range(len(perm)) for s in (0, 1)])
    return G[np.ix_(r, r)]


# 自检：随机特征的 CV-R² 应在 0 附近或为负
_plan = make_plan(40); _Y = np.random.default_rng(3).standard_normal((80, 50)); _X = np.random.default_rng(4).standard_normal((80, 30))
print(f"自检：随机特征 CV-R² = {run_cv(gram(_X), dict(G=gram(_Y), parts={}), _plan, parts=False)['r2']:+.4f}（应 ≤ ~0）")


def run_cv_rank(K, G, plan, ranks=(1, 2, 3)):
    # 降秩 ridge：α 按 run_cv 的内层 CV（满秩）选；再在训练折里把拟合值投到前 r 个输出方向
    # Ŷ_va,r = W Y P_r，P_r = V_r V_rᵀ，V_r = Yᵀ H A_r Σ_r⁻¹（H = 训练折 hat，A, Σ² = eig(H G H)）
    n = K.shape[0]
    sse, sst = {r: np.zeros(n) for r in ranks}, np.zeros(n)
    for (tr, va), inner in plan:
        tot = np.zeros(len(ALPHA_REL))
        for itr, iva in inner:
            for j, W in enumerate(weights(K, itr, iva, ALPHA_REL)):
                tot[j] += row_sse(W, G, itr, iva).sum()
        a = ALPHA_REL[int(np.argmin(tot))]
        Ktt, Kvt = centred_pair(K, tr, va)
        lam, Q = np.linalg.eigh(Ktt); lam = np.clip(lam, 0, None); scale = np.trace(Ktt) / len(tr)
        W = (Kvt @ Q / (lam + a * scale)) @ Q.T
        H = (Q * (lam / (lam + a * scale))) @ Q.T
        Gtt, _ = centred_pair(G, tr, va)
        mu, A = np.linalg.eigh(H @ Gtt @ H); mu, A = mu[::-1], A[:, ::-1]
        sst[va] = centred_vv_diag(G, tr, va)
        for r in ranks:
            Cw = W @ Gtt @ H @ (A[:, :r] / mu[:r]) @ A[:, :r].T @ H
            sse[r][va] = row_sse(Cw, G, tr, va)
    return {r: float(1 - sse[r].sum() / sst.sum()) for r in ranks}

In [ ]:
# ==========================================================================
# Cell 3 — 数据 · DTF · 形状输入（load_rows / sonicom_geometry / symare_geometry 与 Task 1 Cell 5 逐字相同）
# ==========================================================================
def load_rows(tag, ids):
    # 每人两行（左耳, 镜像后的右耳）；每个数组按 manifest 核对 SHA-256
    d = f'{DATA}/{tag}_48k'
    man = {m['id']: m for m in json.load(open(f'{d}/manifest.json'))}
    X = np.empty((2 * len(ids), N_BINS, 3072), np.float32)
    for k, s in enumerate(ids):
        for side in ('left', 'right'):
            f = f'{d}/npy_{side}/{man[s][side]}'
            assert sha256(f) == man[s][f'{side}_sha256'], f
            a = np.load(f)[:N_BINS]
            X[2 * k + (side == 'right')] = a[:, MIRROR] if side == 'right' else a
    return X


def dtf(X):
    # bin 1–85，仰角 ≥ −45°，每只耳每个 bin 减去方向平均（Task 1 的 DTF）
    D = X[:, 1:, :][:, :, MEAS_DIRS]
    return (D - D.mean(2, keepdims=True)).astype(np.float32)


def grams(Ds, Dm):
    # 仿真–仿真、实测–实测、仿真–实测的 Gram，总体和三个频段（float64，逐 bin 累加）
    n = len(Ds)
    out = {k: {b: np.zeros((n, n)) for b in ['all', *BANDS]} for k in ('ss', 'mm', 'sm')}
    for j in range(Ds.shape[1]):
        a, b = Ds[:, j].astype(np.float64), Dm[:, j].astype(np.float64)
        g = {'ss': a @ a.T, 'mm': b @ b.T, 'sm': a @ b.T}
        for k in out:
            out[k]['all'] += g[k]
            for band, m in BAND_DTF.items():
                if m[j]:
                    out[k][band] += g[k]
    return out


def sonicom_geometry(dev):
    # FFM_3_2_1 Cell 4/5：9 个笨几何量 + 耳廓 RFF（5 个 σ），按耳一行；右耳镜像
    man = {f"SON_{r['subject']}": r for r in csv.DictReader(open(f'{SAMPLES}/manifest.csv'))}
    flip = np.array([1.0, -1.0, 1.0])
    G0T = torch.as_tensor(np.random.default_rng(12345).standard_normal((M_RFF, 3)).T, dtype=torch.float32, device=device)

    def rff(P):
        base = 2 * np.pi * (torch.as_tensor(P, dtype=torch.float32, device=device) @ G0T)
        return {sg: torch.cat([torch.cos(sg * base).mean(0), torch.sin(sg * base).mean(0)]).double().cpu().numpy()
                for sg in SIGMAS}

    pct = lambda a, q: float(np.percentile(a, q))
    simple, ear = [], []
    for s in dev:
        d = np.load(f'{SAMPLES}/{s[4:]}.npz')
        head = d['head_pts'][:20000].astype(np.float64)
        head = head[head[:, 2] > -75.0]
        for side in (0, 1):
            if side == 0:
                pts, canal, area, hd = d['earL_pts'], d['ear_left_mm'].astype(np.float64), float(man[s]['crop_area_L_mm2']), head
            else:
                pts, canal, area, hd = d['earR_pts'], d['ear_right_mm'].astype(np.float64) * flip, float(man[s]['crop_area_R_mm2']), head * flip
            assert canal[1] > 0
            q = pts[:20000].astype(np.float64) - canal
            out = q[q[:, 1] > 3.0]
            simple.append([canal[0], canal[1], canal[2], pct(q[:, 1], 95),
                           pct(out[:, 2], 95) - pct(out[:, 2], 5), pct(out[:, 0], 95) - pct(out[:, 0], 5),
                           area, pct(hd[:, 0], 99.5) - pct(hd[:, 0], 0.5), pct(hd[:, 2], 99.9)])
            ear.append(rff(q))
    blocks = {'simple': np.array(simple)}
    for sg in SIGMAS:
        blocks[f'ear_s{sg}'] = np.stack([f[sg] for f in ear])
    GB = {k: (lambda Zx: Zx @ Zx.T / Zx.shape[1])(zscore(X)) for k, X in blocks.items()}
    return sum(GB[k] for k in blocks)


SON_NAMES = ['half inter-canal distance', 'head length', 'head height', 'canal fore-aft', 'canal up-down',
             'pinna protrusion', 'pinna height', 'pinna length', 'ear-crop area']
SON_UNITS = ['mm'] * 8 + ['mm²']
SON_BLOCK = {'head': [0, 1, 2], 'ear': [3, 4, 5, 6, 7, 8]}


def sonicom_measures(dev):
    # 与 sonicom_geometry 的 9 个 simple 量相同（同一段代码），按 SON_NAMES 重新排序：头在前，耳在后
    man = {f"SON_{r['subject']}": r for r in csv.DictReader(open(f'{SAMPLES}/manifest.csv'))}
    flip = np.array([1.0, -1.0, 1.0])
    pct = lambda a, q: float(np.percentile(a, q))
    rows = []
    for s in dev:
        d = np.load(f'{SAMPLES}/{s[4:]}.npz')
        head = d['head_pts'][:20000].astype(np.float64)
        head = head[head[:, 2] > -75.0]
        for side in (0, 1):
            if side == 0:
                pts, canal, area, hd = d['earL_pts'], d['ear_left_mm'].astype(np.float64), float(man[s]['crop_area_L_mm2']), head
            else:
                pts, canal, area, hd = d['earR_pts'], d['ear_right_mm'].astype(np.float64) * flip, float(man[s]['crop_area_R_mm2']), head * flip
            q = pts[:20000].astype(np.float64) - canal
            out = q[q[:, 1] > 3.0]
            rows.append([canal[1], pct(hd[:, 0], 99.5) - pct(hd[:, 0], 0.5), pct(hd[:, 2], 99.9),
                         canal[0], canal[2], pct(q[:, 1], 95),
                         pct(out[:, 2], 95) - pct(out[:, 2], 5), pct(out[:, 0], 95) - pct(out[:, 0], 5), area])
    return np.array(rows)


EAR_P, MIDSAG, EAR_N, PAIR_N, PAIR_P = list(range(13)), list(range(13, 18)), list(range(18, 31)), 31, 32
SYM_HEAD = MIDSAG + [PAIR_P, PAIR_N]          # 中矢面 5 点 + 同侧眼 + 对侧眼（31/32 是左右眼：前方 83 mm、上方 20 mm、±33 mm）


def symare_landmarks(dev):
    # 共同头坐标系（原点约在两耳中点，+x 前，+y 左，+z 上）里的原始 mm 坐标；右耳那一行用整体镜像的构型
    mat = sio.loadmat(PATHS['landmarks'], struct_as_record=False, squeeze_me=True)
    LM = np.stack(list(mat['A'])).astype(np.float64)
    assert LM.shape == (61, 33, 3) and not np.isnan(LM).any()

    def mirror_config(P):
        Q = P.copy(); Q[:, 1] *= -1
        Q[EAR_P], Q[EAR_N] = P[EAR_N] * [1, -1, 1], P[EAR_P] * [1, -1, 1]
        Q[PAIR_P], Q[PAIR_N] = P[PAIR_N] * [1, -1, 1], P[PAIR_P] * [1, -1, 1]
        return Q

    rows = []
    for s in dev:
        P0 = LM[int(s[-2:]) - 1]
        for c in (P0, mirror_config(P0)):
            rows.append(np.concatenate([c[SYM_HEAD].ravel(), c[EAR_P].ravel()]))
    return np.array(rows)


SYM_NAMES = [f'{n} {a}' for n in ['midsag 13', 'midsag 14 (top)', 'midsag 15 (front)', 'midsag 16 (chin)', 'midsag 17 (back)', 'eye ipsi', 'eye contra'] for a in 'xyz'] + \
            [f'ear {i} {a}' for i in range(13) for a in 'xyz']
SYM_BLOCK = {'head': list(range(21)), 'ear': list(range(21, 60))}


def symare_geometry(dev):
    # FFM_2_8：同侧 13 个耳廓点 + 5 个中矢面点 + 同侧眼点（31/32 是左右眼，不是肩），按耳构型（右耳=整体镜像）
    mat = sio.loadmat(PATHS['landmarks'], struct_as_record=False, squeeze_me=True)
    LM = np.stack(list(mat['A'])).astype(np.float64)
    assert LM.shape == (61, 33, 3) and not np.isnan(LM).any()
    EAR_P, MIDSAG, EAR_N, PAIR_N, PAIR_P = list(range(13)), list(range(13, 18)), list(range(18, 31)), 31, 32
    IPSI = EAR_P + MIDSAG + [PAIR_P]

    def mirror_config(P):
        Q = P.copy(); Q[:, 1] *= -1
        Q[EAR_P], Q[EAR_N] = P[EAR_N] * [1, -1, 1], P[EAR_P] * [1, -1, 1]
        Q[PAIR_P], Q[PAIR_N] = P[PAIR_N] * [1, -1, 1], P[PAIR_P] * [1, -1, 1]
        return Q

    rows = []
    for s in dev:
        P0 = LM[int(s[-2:]) - 1]                 # SYM_HAxx -> 第 xx-1 行；+y 是左耳（FFM_2_8 Cell 3b 声学判定）
        for c in (P0, mirror_config(P0)):
            c = c[IPSI]
            rows.append((c - c.mean(0)).ravel())
    X = zscore(np.array(rows))
    return X @ X.T / X.shape[1]

In [ ]:
# ==========================================================================
# Cell 4 — 分析函数（两个库共用）
# ==========================================================================
per_subject = lambda v: v.reshape(-1, 2).sum(1)


def subject_rows(subj):
    return np.ravel(np.column_stack([2 * subj, 2 * subj + 1]))


def check_gate(name, geo, Gm):
    r = run_cv(unit_trace(geo), dict(G=Gm['all'], parts={}), make_plan(len(geo) // 2), parts=False)['r2']
    print(f'复现检查：Task 1 几何 -> 实测 DTF  CV-R² {r:+.4f}（Task 1 {GATE[name]:+.4f}）')
    assert abs(r - GATE[name]) < 0.002, '数据、折或引擎与 Task 1 不同 —— 停下来查'
    return r


def predict(inputs, G, plan, rng):
    R, T = {}, {d: dict(G=G[k]['all'], parts={b: G[k][b] for b in BANDS}) for d, k in (('sim', 'ss'), ('meas', 'mm'))}
    for dom in ('sim', 'meas'):
        for blk, Xc in inputs.items():
            R[f'{blk} -> {dom}'] = run_cv(unit_trace(Xc @ Xc.T), T[dom], plan)
            r = R[f'{blk} -> {dom}']
            edge = sum(c in (0, len(ALPHA_REL) - 1) for c in r['alpha_idx'])
            print(f'  {blk:>4} -> {dom:<5} CV-R² {r["r2"]:+.4f} | ' + '  '.join(f'{b} {r["parts"][b]:+.3f}' for b in BANDS)
                  + (f'  （{edge} 折的 α 在网格边上）' if edge else ''))
    ranks = {dom: run_cv_rank(unit_trace(inputs['all'] @ inputs['all'].T), T[dom]['G'], plan) for dom in ('sim', 'meas')}
    for dom in ('sim', 'meas'):
        print(f'  降秩 all -> {dom:<5} ' + '  '.join(f'秩 {k}: {v:+.4f}' for k, v in ranks[dom].items())
              + f'  | 满秩 {R[f"all -> {dom}"]["r2"]:+.4f}')
    perm = {}
    Kall = unit_trace(inputs['all'] @ inputs['all'].T)
    for dom in ('sim', 'meas'):
        null = np.array([run_cv(permuted(Kall, rng.permutation(len(Kall) // 2)), T[dom], plan, parts=False)['r2']
                         for _ in range(N_PERM_CV)])
        obs = R[f'all -> {dom}']['r2']
        perm[dom] = dict(p=float((1 + (null >= obs).sum()) / (1 + N_PERM_CV)), null_p95=float(np.percentile(null, 95)))
        print(f'  置换 all -> {dom:<5} p = {perm[dom]["p"]:.3f}（零分布 95 分位 {perm[dom]["null_p95"]:+.4f}）')
    r2_of = lambda lbl, i: 1 - per_subject(R[lbl]['sse'])[i].sum() / per_subject(R[lbl]['sst'])[i].sum()
    stats = {'ear added (sim): all - head': lambda i: r2_of('all -> sim', i) - r2_of('head -> sim', i),
             'ear added (meas): all - head': lambda i: r2_of('all -> meas', i) - r2_of('head -> meas', i),
             'all: sim - meas': lambda i: r2_of('all -> sim', i) - r2_of('all -> meas', i)}
    ns, CI = len(Kall) // 2, {}
    for k, f in stats.items():
        bs = np.array([f(rng.integers(0, ns, ns)) for _ in range(N_BOOT)])
        CI[k] = dict(obs=float(f(np.arange(ns))), lo=float(np.percentile(bs, 2.5)), hi=float(np.percentile(bs, 97.5)))
        print(f'  bootstrap {k:<30} {CI[k]["obs"]:+.4f}  95% CI [{CI[k]["lo"]:+.4f}, {CI[k]["hi"]:+.4f}]')
    return R, ranks, perm, CI


def ridge_M(Xc, lam_rel):
    lam = lam_rel * np.trace(Xc.T @ Xc) / len(Xc)          # 与 run_cv 的 α 标度相同（相对 trace / n）
    return np.linalg.inv(Xc.T @ Xc + lam * np.eye(Xc.shape[1]))


def top_modes(Xc, M, G, k):
    H = Xc @ M @ Xc.T
    mu, A = np.linalg.eigh(H @ G @ H)
    mu, A = mu[::-1][:k], A[:, ::-1][:, :k]
    P = Xc.T @ A / np.sqrt(len(Xc))                         # 形状变化：得分 +1 SD 时人群平均的形状变化
    P *= np.sign(P[np.argmax(np.abs(P), 0), np.arange(k)])  # 符号：最大分量为正
    return mu, P, H


def response_stats(c, G, idx=None):
    g = (lambda M: M) if idx is None else (lambda M: M[np.ix_(idx, idx)])
    out = {}
    for b in ['all', *BANDS]:
        d = len(DIRS) * (len(DTF_F) if b == 'all' else int(BAND_DTF[b].sum()))
        ss, mm, sm = c @ g(G['ss'][b]) @ c, c @ g(G['mm'][b]) @ c, c @ g(G['sm'][b]) @ c
        out[b] = dict(gain_sim_db=float(np.sqrt(ss / d)), gain_meas_db=float(np.sqrt(mm / d)),
                      cos=float(sm / np.sqrt(ss * mm)), ratio=float(np.sqrt(mm / ss)))
    return out


def modes(Xc, G, lam_rel, rng, scale):
    n, ns = len(Xc), len(Xc) // 2
    M = ridge_M(Xc, lam_rel)
    mu, P, H = top_modes(Xc, M, G['ss']['all'], N_MODES)
    mu_m, P_m, _ = top_modes(Xc, M, G['mm']['all'], N_MODES)
    fit_total = np.trace(H @ G['ss']['all'] @ H)
    res = []
    for k in range(N_MODES):
        c = Xc @ M @ P[:, k]                                  # 两个映射对形状变化 p_k 的响应 = Yᵀ c
        st = response_stats(c, G)
        # 置换（余弦）：把实测按人打乱，仿真不动
        null_cos = []
        for _ in range(N_PERM):
            pr = subject_rows(rng.permutation(ns))
            c2 = np.empty(n); c2[pr] = c
            null_cos.append(c @ G['sm']['all'] @ c2 / np.sqrt((c @ G['ss']['all'] @ c) * (c2 @ G['mm']['all'] @ c2)))
        null_cos = np.array(null_cos)
        # bootstrap：按人重抽，形状变化 p_k 固定；稳定性 = 重新求出的模式 k 与 p_k 的 |余弦|
        bs, stab = [], []
        for _ in range(N_BOOT):
            idx = subject_rows(rng.integers(0, ns, ns))
            Xb = Xc[idx] - Xc[idx].mean(0)
            Mb = ridge_M(Xb, lam_rel)
            bs.append(response_stats(Xb @ Mb @ P[:, k], G, idx))
            Pb = top_modes(Xb, Mb, G['ss']['all'][np.ix_(idx, idx)], N_MODES)[1]
            stab.append(abs(Pb[:, k] @ P[:, k]) / (np.linalg.norm(Pb[:, k]) * np.linalg.norm(P[:, k])))
        ci = lambda key, b='all': [float(np.percentile([x[b][key] for x in bs], q)) for q in (2.5, 97.5)]
        row = dict(mode=k + 1, share_of_sim_fit=float(mu[k] / fit_total), shape_change=P[:, k].tolist(),
                   shape_change_units=(P[:, k] * scale).tolist(), stats=st,
                   cos_perm_p=float((1 + (null_cos >= st['all']['cos']).sum()) / (1 + N_PERM)),
                   cos_null_p95=float(np.percentile(null_cos, 95)),
                   ci={b: dict(cos=ci('cos', b), ratio=ci('ratio', b), gain_sim_db=ci('gain_sim_db', b),
                               gain_meas_db=ci('gain_meas_db', b)) for b in ['all', *BANDS]},
                   stability_median=float(np.median(stab)), stability_p5=float(np.percentile(stab, 5)),
                   meas_mode_same_index_abs_cos=float(abs(P_m[:, k] @ P[:, k]) / (np.linalg.norm(P_m[:, k]) * np.linalg.norm(P[:, k]))),
                   c=c)
        res.append(row)
        a = st['all']
        print(f'  模式 {k + 1}：占仿真拟合 {row["share_of_sim_fit"]:.0%} | +1 SD 的响应 rms：仿真 {a["gain_sim_db"]:.3f} dB，'
              f'实测 {a["gain_meas_db"]:.3f} dB，比值 {a["ratio"]:.2f} {[round(x, 2) for x in row["ci"]["all"]["ratio"]]} | '
              f'余弦 {a["cos"]:+.3f} {[round(x, 3) for x in row["ci"]["all"]["cos"]]}，置换 p = {row["cos_perm_p"]:.3f} | '
              f'稳定性 {row["stability_median"]:.2f}（5% {row["stability_p5"]:.2f}）')
        print('        分频段 余弦 / 比值：' + '  '.join(f'{b} {st[b]["cos"]:+.2f} / {st[b]["ratio"]:.2f}' for b in BANDS))
    # 仿真模式 1 的强度：把形状按人打乱，α 固定
    null_mu = []
    for _ in range(N_PERM):
        pr = subject_rows(rng.permutation(ns))
        Gp = G['ss']['all'][np.ix_(pr, pr)]
        null_mu.append(np.linalg.eigvalsh(H @ Gp @ H)[-1])
    p_mu = float((1 + (np.array(null_mu) >= mu[0]).sum()) / (1 + N_PERM))
    print(f'  仿真模式 1 的强度：置换 p = {p_mu:.3f} | 实测映射自己的前 3 个模式与仿真模式的 |余弦|（同序号）'
          f'{[round(r["meas_mode_same_index_abs_cos"], 2) for r in res]}')
    return dict(lam_rel=float(lam_rel), mode1_strength_perm_p=p_mu, sim_fit_share=[float(m / fit_total) for m in mu],
                meas_fit_share=[float(m / np.trace(H @ G['mm']['all'] @ H)) for m in mu_m], modes=res)


def analyse(name, inputs, scale, names, units, G, plan, rng):
    print(f'\n---------- {name}：预测 ----------')
    R, ranks, perm, CI = predict(inputs, G, plan, rng)
    a_idx = int(np.round(np.median(R['all -> sim']['alpha_idx'])))
    print(f'\n---------- {name}：模式（α = ALPHA_REL[{a_idx}] = {ALPHA_REL[a_idx]:.3g}，两个域相同）----------')
    MO = modes(inputs['all'], G, ALPHA_REL[a_idx], rng, scale)
    for r in MO['modes'][:2]:
        top = np.argsort(-np.abs(r['shape_change']))[:5]
        print(f'  模式 {r["mode"]} 的形状变化（+1 SD，前 5 项）：' +
              '；'.join(f'{names[j]} {r["shape_change_units"][j]:+.2f} {units[j]}' for j in top))
    return dict(cv={l: dict(r2=r['r2'], bands=r['parts'], alpha_idx=r['alpha_idx']) for l, r in R.items()},
                ranks=ranks, permutation=perm, bootstrap=CI, alpha_idx_modes=a_idx, modes=MO,
                input_names=names, input_units=units, input_scale=scale.tolist())

In [ ]:
# ==========================================================================
# Cell 5 — SONICOM（164 人，折与 Task 1 / FFM_3_2_1 相同）
# ==========================================================================
L = np.load(PATHS['old_latents'])
DEV_S = [str(s) for s in L['eval_sids']]
O = np.load(PATHS['old_oof'], allow_pickle=True)
assert [str(s) for s in O['dev']] == DEV_S and len(DEV_S) == 164
assert not set(DEV_S) & set(SPLIT['SONICOM']['test'])
PLAN_S = make_plan(len(DEV_S))
fold = np.full(len(DEV_S), -1)
for f, ((tr, va), _) in enumerate(PLAN_S):
    fold[np.unique(va // 2)] = f
assert np.array_equal(fold, O['fold_of_subject'])
RNG = np.random.default_rng(SEED + 3)

t0 = time.time()
Ds = dtf(load_rows('SONICOM_SIM', DEV_S)); Dm = dtf(load_rows('SONICOM', DEV_S))
G_S = grams(Ds, Dm)
print(f'SONICOM：DTF 和 Gram {time.time() - t0:.0f}s')
GATE_S = check_gate('SONICOM', sonicom_geometry(DEV_S), G_S['mm'])

Xs = sonicom_measures(DEV_S)
assert Xs.shape == (328, 9) and np.isfinite(Xs).all()
sd_s = Xs.std(0)
Z = (Xs - Xs.mean(0)) / sd_s                               # 按列标准化：形状变化以 SD 计，乘 sd 回到 mm
IN_S = {'head': Z[:, SON_BLOCK['head']], 'ear': Z[:, SON_BLOCK['ear']], 'all': Z}
for n_, m_, s_ in zip(SON_NAMES, Xs.mean(0), sd_s):
    print(f'  {n_:<26} 均值 {m_:9.1f}  SD {s_:7.2f}')
RES_S = analyse('SONICOM', IN_S, sd_s, SON_NAMES, SON_UNITS, G_S, PLAN_S, RNG)
RES_S['gate'] = GATE_S
RESP_S = {k: {d: np.tensordot(r['c'], D, axes=(0, 0)) for d, D in (('sim', Ds), ('meas', Dm))}
          for k, r in enumerate(RES_S['modes']['modes'])}
del Ds, Dm

In [ ]:
# ==========================================================================
# Cell 6 — SYMARE（46 人）
# ==========================================================================
DEV_Y = sorted(SPLIT['SYMARE']['train'] + SPLIT['SYMARE']['val'])
assert len(DEV_Y) == 46 and not set(DEV_Y) & set(SPLIT['SYMARE']['test'])
PLAN_Y = make_plan(len(DEV_Y))
Dsy = dtf(load_rows('SYMARE_BEM', DEV_Y)); Dmy = dtf(load_rows('SYMARE', DEV_Y))
G_Y = grams(Dsy, Dmy)
GATE_Y = check_gate('SYMARE', symare_geometry(DEV_Y), G_Y['mm'])

Xy = symare_landmarks(DEV_Y)
assert Xy.shape == (92, 60) and np.isfinite(Xy).all()
Zy = Xy - Xy.mean(0)                                       # mm，不标准化：形状变化直接是 mm
IN_Y = {'head': Zy[:, SYM_BLOCK['head']], 'ear': Zy[:, SYM_BLOCK['ear']], 'all': Zy}
EAR_MEAN_Y = Xy[:, 21:].mean(0).reshape(13, 3)
RES_Y = analyse('SYMARE', IN_Y, np.ones(60), SYM_NAMES, ['mm'] * 60, G_Y, PLAN_Y, RNG)
RES_Y['gate'] = GATE_Y
RESP_Y = {k: {d: np.tensordot(r['c'], D, axes=(0, 0)) for d, D in (('sim', Dsy), ('meas', Dmy))}
          for k, r in enumerate(RES_Y['modes']['modes'])}

In [ ]:
# ==========================================================================
# Cell 7 — 保存 · 图
# ==========================================================================
import matplotlib.pyplot as plt


def strip(res):
    out = json.loads(json.dumps({k: v for k, v in res.items() if k != 'modes'}, default=float))
    mo = dict(res['modes']); mo['modes'] = [{k: v for k, v in r.items() if k != 'c'} for r in mo['modes']]
    out['modes'] = json.loads(json.dumps(mo, default=float))
    return out


RES = dict(created=time.strftime('%Y-%m-%d %H:%M:%S'), notebook='task2/Task2_ShapeModes.ipynb',
           n_perm_cv=N_PERM_CV, n_perm=N_PERM, n_boot=N_BOOT, databases={'SONICOM': strip(RES_S), 'SYMARE': strip(RES_Y)})
json.dump(RES, open(f'{OUT}/task2_results.json', 'w'), indent=1)
np.savez_compressed(f'{OUT}/task2_responses.npz', freqs_hz=DTF_F, dirs=DIRS,
                    **{f'{db}_mode{k + 1}_{d}': R_[k][d].astype(np.float32) for db, R_ in (('SONICOM', RESP_S), ('SYMARE', RESP_Y))
                       for k in R_ for d in ('sim', 'meas')})


def plane(r, P):
    return np.array([r[:, j] for _, j in P]).T                # (85 bins, 方向)


def draw(db, res, resp, shape_panel, fname):
    nm = 2
    fig, ax = plt.subplots(nm, 6, figsize=(30, 4.6 * nm), gridspec_kw=dict(width_ratios=[1.2, 1, 1, 1, 1, 1]))
    for k in range(nm):
        m = res['modes']['modes'][k]; r = resp[k]
        shape_panel(ax[k, 0], m)
        ax[k, 0].set_title(f'{db} mode {k + 1}: shape change at +1 SD\n({m["share_of_sim_fit"]:.0%} of the simulated fit)', fontsize=10)
        for d, c in (('sim', 'C3'), ('meas', 'C0')):
            ax[k, 1].plot(DTF_F / 1000, np.sqrt((r[d] ** 2).mean(1)), c=c, label=d)
        a = m['stats']['all']
        ax[k, 1].set_title(f'rms over directions (dB)\ncos {a["cos"]:+.2f}, meas/sim {a["ratio"]:.2f}', fontsize=10)
        ax[k, 1].legend(fontsize=8); ax[k, 1].grid(alpha=.3); ax[k, 1].set_xlabel('kHz')
        v = np.percentile(np.abs(np.concatenate([plane(r['sim'], H_PLANE), plane(r['sim'], M_PLANE)], 1)), 99)
        for j, (d, P, xs, lab) in enumerate((('sim', H_PLANE, 'azimuth (deg, + = ipsilateral)', 'horizontal'),
                                             ('meas', H_PLANE, 'azimuth (deg, + = ipsilateral)', 'horizontal'),
                                             ('sim', M_PLANE, 'polar angle (deg: −45 front-below … 90 up … 225 back-below)', 'median'),
                                             ('meas', M_PLANE, 'polar angle (deg: −45 front-below … 90 up … 225 back-below)', 'median'))):
            x = [a_ for a_, _ in P]
            im = ax[k, 2 + j].pcolormesh(x, DTF_F / 1000, plane(r[d], P), cmap='RdBu_r', vmin=-v, vmax=v, shading='nearest')
            ax[k, 2 + j].set_title(f'{d}: {lab} plane (dB per +1 SD)', fontsize=10); ax[k, 2 + j].set_xlabel(xs, fontsize=8)
            ax[k, 2 + j].set_ylabel('kHz')
        fig.colorbar(im, ax=ax[k, 2:].tolist(), shrink=0.8)
    plt.savefig(f'{OUT}/{fname}', dpi=110, bbox_inches='tight'); plt.show()


def son_shape(a, m):
    u = np.array(m['shape_change'])
    a.barh(range(9), u, color=['C1'] * 3 + ['C2'] * 6); a.set_yticks(range(9))
    a.set_yticklabels([f'{n} ({x:+.1f} {un})' for n, x, un in zip(SON_NAMES, m['shape_change_units'], SON_UNITS)], fontsize=8)
    a.invert_yaxis(); a.axvline(0, c='k', lw=.6); a.set_xlabel('SD of each measure (orange: head, green: ear)')


def sym_shape(a, m):
    u = np.array(m['shape_change']); e = u[21:].reshape(13, 3); h = u[:21].reshape(7, 3)
    a.scatter(EAR_MEAN_Y[:, 0], EAR_MEAN_Y[:, 2], c='k', s=12)
    q = a.quiver(EAR_MEAN_Y[:, 0], EAR_MEAN_Y[:, 2], e[:, 0], e[:, 2], e[:, 1], cmap='coolwarm', angles='xy',
                 scale_units='xy', scale=0.2)
    for i, (x, z) in enumerate(EAR_MEAN_Y[:, [0, 2]]):
        a.annotate(str(i), (x, z), fontsize=7, xytext=(2, 2), textcoords='offset points')
    a.set_aspect('equal'); a.set_xlabel('x (mm, forward)'); a.set_ylabel('z (mm, up)')
    a.text(0.02, 0.02, f'arrows x5; colour = lateral (y) change\nhead points: rms {np.sqrt((h ** 2).sum(1)).mean():.2f} mm, '
                       f'ear points: rms {np.sqrt((e ** 2).sum(1)).mean():.2f} mm', transform=a.transAxes, fontsize=7)


draw('SONICOM', RES_S, RESP_S, son_shape, 'task2_modes_sonicom.png')
draw('SYMARE', RES_Y, RESP_Y, sym_shape, 'task2_modes_symare.png')

print(f"\n{'':<24}{'SONICOM sim':>12}{'meas':>9}{'SYMARE sim':>12}{'meas':>9}")
for blk in ('head', 'ear', 'all'):
    print(f'{blk + " (CV-R²)":<24}' + ''.join(f"{R_['cv'][f'{blk} -> {d}']['r2']:>+{w}.4f}" for R_ in (RES['databases']['SONICOM'], RES['databases']['SYMARE'])
                                           for d, w in (('sim', 12), ('meas', 9))))
print(f'\n结果 -> {OUT}/task2_results.json, task2_responses.npz, task2_modes_sonicom.png, task2_modes_symare.png')